## Lasso Regression

In this notebook, we build a Lasso(L1) Regression model to predict medical insurance charges based on customer information. The model learns the relationship between the input features and the target variable by fitting the best possible linear equation that minimizes prediction error.

The workflow includes:
- Loading the dataset
- Data preprocessing
- Training the LassoCv Regression model
- Making predictions
- Evaluating model performance using MAE, MSE, RMSE, and R² Score
- Saving the trained model

In [1]:
import pandas as pd
df=pd.read_csv("../data/insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## X and y value sepration

In [22]:
X=df.drop("charges",axis=1)
y=df["charges"]

## Train and test split of X and y

In [23]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42)

## Binary encoding of sex and smoker using pd.map in X_train and X_test

In [24]:
X_train["sex"]=X_train["sex"].map({"female":0,"male":1})
X_test["sex"]=X_test["sex"].map({"female":0,"male":1})
X_train["smoker"]=X_train["smoker"].map({"yes":1,"no":0})
X_test["smoker"]=X_test["smoker"].map({"yes":1,"no":0})


## One Hot Encoding on X_train and X_test for region section section

In [25]:
X_train=pd.get_dummies(X_train,columns=["region"],drop_first=True,dtype=int)
X_test=pd.get_dummies(X_test,columns=["region"],drop_first=True,dtype=int)


## Scaling of the children ,bmi and age columns 

In [16]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train[["age","bmi","children"]]=scaler.fit_transform(
    X_train[["age","bmi","children"]])
X_test[["age","bmi","children"]]=scaler.transform(
    X_test[["age","bmi","children"]])

,age,sex,bmi,children,smoker,region
764,45.0,NaN,25.175,2.000000e+00,NaN,northeast
887,36.0,NaN,30.020,-6.640586e-18,NaN,northwest
890,64.0,NaN,26.885,-6.640586e-18,NaN,northwest
1293,46.0,NaN,25.745,3.000000e+00,NaN,northwest
259,19.0,NaN,31.920,-6.640586e-18,NaN,northwest
...,...,...,...,...,...,...
109,63.0,NaN,35.090,-6.640586e-18,NaN,southeast
575,58.0,NaN,27.170,-6.640586e-18,NaN,northwest
535,38.0,NaN,28.025,1.000000e+00,NaN,northeast
543,54.0,NaN,47.410,-6.640586e-18,NaN,southeast


In [34]:
from sklearn.linear_model import LassoCV
alpha_=[0.001, 0.01, 0.1, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
lasso_cv_model=LassoCV(
    alphas=alpha_,
    cv=5,
    max_iter=10000,
    random_state=42
)
lasso_cv_model.fit(X_train,y_train)
print("best alpha:",lasso_cv_model.alpha_)

best alpha: 100.0


In [36]:
y_pred=lasso_cv_model.predict(X_test)
y_pred

array([ 8628.41043175,  7171.20026025, 36366.20380711,  9432.64402776,
       26476.05704499, 11205.78438   ,   379.15495786, 16904.07548234,
         987.02637982, 11157.66553559, 27939.53480663,  9362.89068471,
        5601.58537277, 37868.36477286, 39900.76973114, 36653.85029034,
       15339.16957332, 35540.52875517,  9455.48973609, 30987.54584034,
        4180.56487803, 10528.06403309,  2940.48622426,  6763.94780577,
       11228.63562835, 12645.54207723, 14968.40443412,  6078.35043024,
        9603.55476417,  2599.90231399,  9413.62687013, 13136.18707113,
        4904.75594744,  3452.19472648,  4966.38839237, 12654.85891223,
        2516.34621415,  9288.05648728, 32760.71254543, 32258.62111452,
        4170.53430513,  4380.41061135, 14534.23385111, 11580.85955   ,
        8990.03149749, 12606.47538549,  5227.86049893,  3584.59814977,
       35027.53186362,  9335.34519905, 16126.58725332,  2712.72920103,
       12261.78663049,  1295.68607624, 13726.25808518, 12161.7804189 ,
      

In [38]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
mae=mean_absolute_error(y_test,y_pred)
print("The mean abosulte error is:",mae);
mse=mean_squared_error(y_test,y_pred)
print("The mean squared error is:",mse);
r2=r2_score(y_test,y_pred)
print("The r2 score is:",r2*100,"%");
n=X_test.shape[0]
p=X_test.shape[1]
adjusted_r2=1-((1-r2)*(n-1))/(n-p-1)
print("the adjusted_r2 is:",adjusted_r2)

The mean abosulte error is: 4268.40263712877
The mean squared error is: 34245283.94567648
The r2 score is: 77.94166585670872 %
the adjusted_r2 is: 0.7726032735035223


In [40]:
y_train_pred=lasso_cv_model.predict(X_train)
from sklearn.metrics import r2_score

r2_train = r2_score(y_train, y_train_pred)
print("Training R² Score:", r2_train)
r2_test = r2_score(y_test, y_pred)
print("Testing R² Score:", r2_test)

Training R² Score: 0.7406181234209634
Testing R² Score: 0.7794166585670872


In [41]:
print(f"Training R² : {r2_train:.4f}")
print(f"Testing R²  : {r2_test:.4f}")

Training R² : 0.7406
Testing R²  : 0.7794


In [43]:
lasso_coef = lasso_cv_model.coef_
print(lasso_coef)
import pandas as pd

coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Lasso Coefficient": lasso_cv_model.coef_
})

print(coef_df)

[  256.12656985     0.           324.83592472   362.95054907
 23041.82021516     0.            -0.            -0.        ]
            Feature  Lasso Coefficient
0               age         256.126570
1               sex           0.000000
2               bmi         324.835925
3          children         362.950549
4            smoker       23041.820215
5  region_northwest           0.000000
6  region_southeast          -0.000000
7  region_southwest          -0.000000


In [44]:
coef_df[coef_df["Lasso Coefficient"] == 0]

,Feature,Lasso Coefficient
1,sex,0.0
5,region_northwest,0.0
6,region_southeast,-0.0
7,region_southwest,-0.0


In [45]:
coef_df[coef_df["Lasso Coefficient"] != 0]

,Feature,Lasso Coefficient
0,age,256.126570
2,bmi,324.835925
3,children,362.950549
4,smoker,23041.820215
